# 04. The reconstruction: the benchmark against CRSP's own figures

**What this notebook does.** It builds the benchmark (the universe with each member weighted by its market value at the start of the month) from the files read in notebooks 02 and 03, and compares the benchmark's return with CRSP's own return of the same list, weighted the same way, with dividends and price-only. That comparison judges gate G1 and expectation E1. It compares the universe's count and summed market value with CRSP's count and total value on the same day, which judges expectation E2. It sets the price-only return beside the index's own price return as S&P publishes it, and does not judge that comparison. And it gathers the data layer's six results in one file.

**Why it comes here.** CRSP computes its return from the same list and the same stock files. If the benchmark built here differs from it by more than two tenths of a percentage point a year, the universe, the returns or the weights are wrong somewhere, and every later result would carry the error.

**Terms used in this notebook.**

- *universe*: the stocks in CRSP's list of index members on the last trading day of the month before.
- *market capitalisation*: price times shares outstanding. The capitalisation at the start of month t is the absolute price times shares outstanding at the end of month t−1.
- *cap-weighted*: weighted by market capitalisation, each stock's weight being its capitalisation divided by the sum over the stocks held. *Value-weighted* is CRSP's word for the same.
- *benchmark*: the cap-weighted universe, with each stock's weight set by its capitalisation at the start of the month, so that nothing unknown when the month begins sets a weight. A member without a capitalisation, or with neither a return nor a delisting in the month, is left out of that month's return and the other weights are rescaled. Both are counted.
- *return with dividends* and *price-only return*: the month's return with dividends reinvested, and without them.
- *CRSP's return*: CRSP's own value-weighted return of the members of its list, computed from its daily data, with dividends (`vwretd`) and price-only (`vwretx`).
- *the index's price return*: S&P's published return of the index without dividends, which CRSP stores beside its own (`sprtrn`). Since 2005 the index weights each stock by its float-adjusted shares, the shares available to the public, which CRSP's share counts are not.
- *CRSP's count* and *CRSP's total value*: the number of index members with a valid price on a day, and the sum of their capitalisations in thousands of dollars, ADRs left out (`totcnt` and `totval`). Each row of CRSP's monthly index file carries them for that month's last trading day.
- *basis point*: one hundredth of a percentage point.
- *mean monthly difference*: the average over the months of the benchmark's return minus CRSP's.
- *tracking error*: the standard deviation of the monthly difference between two returns, times the square root of 12, a yearly figure.
- *usable window*: the months from the first in which the universe's count is within 10 names of CRSP's count, to December 2024, as notebook 02 found it.
- *gate* and *expectation*: tests fixed in advance in `src/slt/constants.py`. A gate that is not met stops the study. An expectation that is not met is reported as a finding.
- *verdict*: the result of a gate or an expectation, met or not met, printed in the same words either way.

**What I expect to see, and where it comes from.** The levels are the constants', committed before any run that judges them.

- *G1*: tracking error against CRSP's value-weighted return with dividends of at most 50 basis points a year over the usable window.
- *E1*: tracking error of at most 20 basis points a year and a mean monthly difference within 2 basis points of zero, with dividends and price-only. CRSP computes its return from the same list and the same file, daily. The differences left are membership changes inside a month, held here to the start of the month, and CRSP's treatment of the day of removal, which concern a small number of names a year, at small weights. So E1, and with it G1, are expected met.
- *E2*: in every month of the usable window the universe's count within 10 names of CRSP's count, and the summed capitalisation within 2 per cent of CRSP's total value in at least 95 per cent of months. Expected met, with one inference written before any run: CRSP's total value leaves out ADRs and the universe filters no share code, so in months with ADR members, which notebook 02 lists by year, the summed capitalisation may sit above CRSP's total value.
- *The index's price return*: until 2005 the index weighted each stock by its full capitalisation, as the benchmark does, so over the months to December 2004 the gap is evidence on membership independent of CRSP's list, removals included, which Compustat's history cannot test. A wider gap is expected from 2005, when the index moved to float-adjusted shares. Neither figure has a verdict.

Running time: about a minute.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
from slt import layer as L
from slt import returns as R
from slt import sectors as S
from slt import universe as U
from slt import validation as V

## 1. The benchmark

The benchmark is built from the monthly panel of notebook 03, rebuilt here from the files, and kept to the usable window. The members left out of a month's return are counted over the window, with the largest number in any one month.

In [ ]:
window = X.read_result("usable_window.json")
first = pd.Period(window["first_month"], "M")
t0 = time.time()
universe = L.load_universe(root)
data = L.load_panel(root, universe)
bench = V.benchmark(data["panel"])
crsp = V.crsp_by_month(universe["index_file"])
b = bench[bench.index >= first]
left_out = {"months": int(len(b)),
            "member_months_without_cap": int(b["members_without_cap"].sum()),
            "most_in_one_month_without_cap": int(b["members_without_cap"].max()),
            "member_months_without_return": int(b["members_without_return"].sum()),
            "most_in_one_month_without_return": int(b["members_without_return"].max())}
print(f"usable window {window['first_month']} to {window['last_month']} | {time.time() - t0:.0f} s")
print(json.dumps(left_out, indent=1))

## 2. Against CRSP's value-weighted return: G1 and E1

Each comparison runs over the months both series have in the usable window. The table printed after the verdicts gives the ten years with the largest tracking error. The full table by year is committed, and no figure of CRSP's by month is. The same two figures over the evaluation window of the later notebooks, July 1999 to December 2024, are printed last and committed in the summary, with no verdict.

In [ ]:
with_div = V.compare(b["benchmark_return"], crsp["crsp_return"])
price_only = V.compare(b["benchmark_return_price_only"], crsp["crsp_return_price_only"])
judged = V.judge_g1_e1(with_div, price_only)
for label, c in (("with dividends", with_div), ("price-only", price_only)):
    print(f"{label}: months {c['months']} | tracking error {c['tracking_error'] * 1e4:.1f} basis points a year"
          f" | mean monthly difference {c['mean_difference'] * 1e4:.2f} basis points | correlation {c['correlation']:.6f}")
print(V.verdict_line("G1", judged["G1"]["met"], f"Tracking error {with_div['tracking_error'] * 1e4:.1f} basis points a year against CRSP's "
      f"value-weighted return with dividends, against a level of at most {C.G1_TE_MAX * 1e4:.0f}."))
print(V.verdict_line("E1", judged["E1"]["met"], f"Tracking error {with_div['tracking_error'] * 1e4:.1f} and {price_only['tracking_error'] * 1e4:.1f} "
      f"basis points a year and mean monthly difference {with_div['mean_difference'] * 1e4:.2f} and {price_only['mean_difference'] * 1e4:.2f} "
      f"basis points, with dividends and price-only, against at most {C.E1_TE_MAX * 1e4:.0f} and within {C.E1_MEAN_DIFF_MAX * 1e4:.0f} of zero."))
parts = [V.compare_by_year(b["benchmark_return"], crsp["crsp_return"]).add_suffix("_with_dividends"),
         V.compare_by_year(b["benchmark_return_price_only"], crsp["crsp_return_price_only"])[["tracking_error", "mean_difference"]].add_suffix("_price_only"),
         V.compare_by_year(b["benchmark_return_price_only"], crsp["index_price_return"])[["tracking_error", "mean_difference"]].add_suffix("_against_index_price")]
by_year = parts[0].join(parts[1:]).rename(columns={"months_with_dividends": "months"}).reset_index()
print(by_year.sort_values("tracking_error_with_dividends", ascending=False).head(10).to_string(index=False))
in_eval = (b.index >= pd.Period(C.EVAL_START, "M")) & (b.index <= pd.Period(C.EVAL_END, "M"))
evaluation_window = {"first_month": C.EVAL_START, "last_month": C.EVAL_END,
                     "with_dividends": V.compare(b.loc[in_eval, "benchmark_return"], crsp["crsp_return"]),
                     "price_only": V.compare(b.loc[in_eval, "benchmark_return_price_only"], crsp["crsp_return_price_only"])}
for label in ("with_dividends", "price_only"):
    c = evaluation_window[label]
    print(f"evaluation window, {label}: months {c['months']} | tracking error {c['tracking_error'] * 1e4:.1f} basis points a year"
          f" | mean monthly difference {c['mean_difference'] * 1e4:.2f} basis points, with no verdict")

## 3. Against the index's price return, with no verdict

The benchmark's price-only return beside the index's own price return, over the whole usable window, over the months to December 2004 and over the months from January 2005. Until 2005 the index weighted each stock by its full capitalisation, as the benchmark does, so the gap to December 2004 comes from membership, prices and share counts, and is evidence on membership independent of CRSP's list, removals included. From 2005 the float adjustment adds to it.

In [ ]:
last_full = pd.Period(C.INDEX_FULL_CAP_LAST_MONTH, "M")
index_price = {"whole_window": V.compare(b["benchmark_return_price_only"], crsp["index_price_return"]),
               "to_december_2004": V.compare(b.loc[b.index <= last_full, "benchmark_return_price_only"], crsp["index_price_return"]),
               "from_january_2005": V.compare(b.loc[b.index > last_full, "benchmark_return_price_only"], crsp["index_price_return"])}
for label, c in index_price.items():
    print(f"{label}: months {c['months']} | tracking error {c['tracking_error'] * 1e4:.1f} basis points a year"
          f" | mean monthly difference {c['mean_difference'] * 1e4:.2f} basis points | correlation {c['correlation']:.6f}")

## 4. Count and value: E2

The universe at the start of month t, which is the membership and the capitalisation on the last trading day of month t−1, against CRSP's count and total value on its row for month t−1, the same day.

In [ ]:
e2 = V.judge_e2(bench, crsp, window["first_month"])
e2_years = V.e2_value_by_year(bench, crsp, window["first_month"])
print(json.dumps(e2, indent=1))
print(V.verdict_line("E2", e2["met"], f"{e2['months_count_outside']} of {e2['months']} months have a count more than {C.E2_COUNT_WITHIN} names "
      f"from CRSP's count, against a level of none, and the summed capitalisation is within {C.E2_VALUE_REL_MAX:.0%} of CRSP's total value "
      f"in {e2['share_value_within']:.2%} of months, against a level of at least {C.E2_VALUE_MONTH_SHARE_MIN:.0%}."))
print(e2_years[e2_years["months_value_outside"] > 0].to_string(index=False))

## 5. The data layer's six verdicts

G2 comes from notebook 02, G3 and G4 from notebook 03, and G1, E1 and E2 from this notebook. If any of the four gates is not met, the study stops at the data layer, which is published alone with the reason.

In [ ]:
g3 = X.read_result("removals_g3.json")["in_daily_file"]
g4 = X.read_result("capitalisation_g4.json")
verdicts = {
    "G1": {"met": judged["G1"]["met"], "tracking_error": with_div["tracking_error"], "level": C.G1_TE_MAX},
    "G2": {"met": window["g2_met"], "years": window["years"], "level": C.G2_USABLE_YEARS_MIN},
    "G3": {"met": g3["g3_met"], "share": g3["share"], "removals": g3["removals"], "level": C.G3_SHARE_MIN},
    "G4": {"met": g4["g4_met"], "share_of_names": g4["share_of_names"], "names": g4["names"], "level": C.G4_NAME_SHARE_MIN},
    "E1": judged["E1"],
    "E2": e2,
}
for name, v in verdicts.items():
    print(f"{name}: {'met' if v['met'] else 'not met'}")
gates = all(verdicts[g]["met"] for g in ("G1", "G2", "G3", "G4"))
print(V.verdict_line("The four gates", gates, "The study goes on to the answer key." if gates else "The study stops at the data layer."))

## 6. What is committed

Four files go into `results/20260928/`: the comparisons by year, the value comparison of E2 by year, a summary of every comparison over the usable window and over the evaluation window, and the six verdicts. Nothing is committed by month. The benchmark's return, the universe's count and its summed capitalisation reproduce CRSP's index file so closely, month by month, that a table of them would be that file under other names, so they stay in memory.

In [ ]:
summary = {
    "written_by": "notebooks/04_reconstruction.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
    "usable_window": {k: window[k] for k in ("first_month", "last_month", "months")},
    "members_left_out": left_out, "with_dividends": with_div, "price_only": price_only,
    "against_index_price_return": index_price, "e2": e2, "evaluation_window": evaluation_window,
}
written = [X.write_results("reconstruction_by_year.csv", frame=by_year),
           X.write_results("e2_value_by_year.csv", frame=e2_years),
           X.write_results("reconstruction_summary.json", record=summary),
           X.write_results("verdicts_milestone1.json", record=verdicts)]
for path in written:
    print("written:", path.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: how closely the benchmark built from the universe reproduces CRSP's own return of the same list, with dividends and price-only, how closely its count and capitalisation match CRSP's on the same day, the verdicts of G1, E1 and E2, and with notebooks 02 and 03 whether the data layer passes its four gates.

What could still be wrong. A close match to CRSP shows that the data layer reproduces CRSP's construction. It does not show that CRSP's list or prices are right, since both sides read the same source. Section 3 is the one comparison with a source of its own: the index S&P publishes, which weighted by full capitalisation until 2005, so its gap to December 2004 tests membership, removals included. From 2005 the index weights by float-adjusted shares and the benchmark by every share outstanding, so the gap grows for a reason other than an error, and section 3 reports it without judging it. And the tracking error is one figure over a long window: a year with a large gap can sit inside a small figure, which is why the table by year is committed beside it.